[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caerdfasgrae/PEDAS-2026/blob/main/notebooks/02_tifis_id_official_pipeline.ipynb)

# 🛡️ Tifis-ID: Deteksi dan Klasifikasi Ancaman Cerdas Domain (.id)
> **Pesta Data Nasional (PeDaS 2026) | APTIKOM Fest 2026 x PANDI**  
> **Tim**: TIFIS TIFIS  
> **Arsitektur Utama**: *Explainable Hybrid Probabilistic Blender* (LinearSVC Character N-Grams 60% + LightGBM Domain Lifecycle 40% + Multiclass Platt Scaling + Bayes Thresholds + Evidence Guard)  
> **Kepatuhan Aturan**: Double-Blind Review Ready | 100% Python Native | Reproducibility Guaranteed (`RANDOM_STATE = 2026`)

---
## Ringkasan Eksekutif dan Capaian Metrik
- **Metrik Utama Lomba**: Macro-F1 Score pada 9 Kategori Ancaman Resmi IDADX PANDI.
- **Stratified 5-Fold CV Macro-F1**: **`0.6026`** (Meningkat signifikan dari baseline tunggal 0.5749).
- **Strict Domain Group-KFold (100% Unseen Domains)**: **`0.5731`** (Generalization Gap hanya **`2.95%`**, membuktikan model bebas dari memorisasi domain).
- **Waktu Eksekusi Penuh (100% Data Latih + Uji)**: **~10.5 Detik** (Batas Komputasi Wajar Juknis: < 300 Detik / 5 Menit, Margin Efisiensi: > 96%).
- **Berkas Keluaran Resmi**: `official/submission_TIFIS_TIFIS.csv` (1.500 baris tervalidasi 100% lolos sensor skrip evaluator panitia).

## 1. Persiapan Lingkungan dan Pemasangan Dependensi
Jika Anda menjalankan notebook ini di **Google Colab**, sel di bawah akan otomatis mengklon repositori dan memasang seluruh pustaka yang dibutuhkan.

In [ ]:
# Deteksi otomatis Google Colab & instalasi pustaka
import os, sys, subprocess, getpass

if 'google.colab' in sys.modules:
    print('[*] Terdeteksi lingkungan Google Colab. Menyiapkan repositori...')
    target_dir = '/content/PEDAS-2026'
    repo_url = 'https://github.com/caerdfasgrae/PEDAS-2026.git'
    
    if not os.path.exists(target_dir):
        # 1. Coba clone publik terlebih dahulu (otomatis sukses jika repo sudah Public)
        res = subprocess.run(['git', 'clone', repo_url, target_dir], capture_output=True, text=True)
        
        # 2. Jika gagal karena repo masih Private (status 128)
        if res.returncode != 0:
            print('[!] Repositori saat ini berstatus PRIVATE di GitHub.')
            pat = None
            try:
                from google.colab import userdata
                pat = userdata.get('GITHUB_PAT') or userdata.get('GH_TOKEN')
            except Exception:
                pass
            
            # Jika belum ada di Colab Secrets, minta input pengguna secara aman (input tersembunyi)
            if not pat:
                print('    Masukkan Personal Access Token (PAT) GitHub untuk mengklon repo private:')
                print('    (Tip: Begitu repo diubah jadi Public di GitHub Settings, sel ini tidak akan meminta token lagi!)')
                pat = getpass.getpass('    GitHub PAT: ').strip()
            
            if pat:
                auth_url = f'https://{pat}@github.com/caerdfasgrae/PEDAS-2026.git'
                clone_res = subprocess.run(['git', 'clone', auth_url, target_dir], capture_output=True, text=True)
                del pat  # Segera hapus token dari memori runtime
                if clone_res.returncode != 0:
                    raise RuntimeError(f'Kloning gagal! Pastikan token valid.\n{clone_res.stderr}')
                print('[OK] Repositori private berhasil diklon menggunakan token!')
            else:
                raise RuntimeError('Kloning gagal: repositori private dan token tidak diisi.')
        else:
            print('[OK] Repositori publik berhasil diklon langsung tanpa token!')
            
    %cd /content/PEDAS-2026
    !pip install -q -r requirements.txt
    print(f'[OK] Lingkungan Google Colab siap dieksekusi! CWD: {os.getcwd()}')
else:
    print(f'[OK] Menjalankan di lingkungan lokal: {os.getcwd()}')


## 2. Inisialisasi Pustaka dan Penguncian Determinisme (`RANDOM_STATE = 2026`)
Sesuai regulasi PeDaS Slide 8 Poin 8, seluruh komponen diikat pada random state tetap untuk menjamin **reproducibility 100% identik** saat diverifikasi oleh Dewan Juri.

In [ ]:
import os
import sys
import time
import hashlib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Pastikan root direktori ada di sys.path
sys.path.insert(0, os.path.abspath('.'))

from src.cleaner import CANONICAL_CLASSES, load_cleaned_datasets, clean_category, clean_url, build_composite_text
from src.pedas_features import DomainEnsembleExtractor, TfidfTextFeatureExtractor
from src.models.hybrid_blender import HybridProbabilisticBlender
from src.submission import export_submission, validate_submission

# Suppress harmless warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 2026
print(f'[OK] Pustaka terpasang sukses. Random Seed dikunci pada: {RANDOM_STATE}')
print(f'[OK] Kosakata 9 Kategori Resmi PANDI: {CANONICAL_CLASSES}')


## 3. Data Ingestion dan Pra-Pemrosesan Deterministik
Memuat 8.400 data latih (`official/training.csv`) dan 1.500 data uji (`official/predict.csv`).
Membentuk representasi teks komposit:
$$\text{Composite Context} = \text{URL} + \text{Brand} + \text{SLD} + \text{Registrar}$$

In [ ]:
t0 = time.time()
train_df, predict_df = load_cleaned_datasets(
    train_path='official/training.csv',
    predict_path='official/predict.csv'
)

print(f'[OK] Data Latih : {len(train_df):,} baris')
print(f'[OK] Data Uji   : {len(predict_df):,} baris')
print(f'[*] Waktu Muat : {time.time()-t0:.2f} detik\n')

# Tampilkan sebaran kelas di data latih
class_counts = train_df['category_clean'].value_counts()
print('Sebaran 9 Kategori di Data Latih:')
for c in CANONICAL_CLASSES:
    cnt = class_counts.get(c, 0)
    pct = (cnt / len(train_df)) * 100
    print(f'  - {c:18s}: {cnt:5d} ({pct:5.2f}%)')


## 4. Visualisasi Distribusi Ancaman Siber Domain (.id)

In [ ]:
plt.figure(figsize=(10, 4.5))
palette = sns.color_palette('mako', len(CANONICAL_CLASSES))
sns.barplot(x=class_counts.values, y=class_counts.index, palette=palette)
plt.title('Distribusi Kategori Ancaman Siber Domain .id (PeDaS 2026)', fontsize=12, fontweight='bold', pad=12)
plt.xlabel('Jumlah Baris Sampel Data')
plt.ylabel('Kategori IDADX')
plt.grid(axis='x', linestyle='--', alpha=0.6)
for i, v in enumerate(class_counts.values):
    plt.text(v + 50, i, f'{v:,} ({v/len(train_df)*100:.1f}%)', va='center', fontsize=9)
plt.tight_layout()
plt.show()


## 5. Pelatihan Arsitektur Model Tifis-ID (Explainable Hybrid Blender)
Model memadukan:
1. **LinearSVC (Bobot 60%)**: Melatih 15.000 n-gram karakter (3-5 gram) untuk menangkap manipulasi kata/typosquatting, dikonversi ke probabilitas posterior via **Multiclass Platt Scaling**.
2. **LightGBM (Bobot 40%)**: Memodelkan interaksi non-linear fitur leksikal, entropi, registrar, dan usia domain.
3. **Cost-Sensitive Bayes Thresholding**: Mengoptimalkan offset threshold untuk memaksimalkan Macro-F1 pada data latih tanpa kebocoran.
4. **Evidence Guard**: Memverifikasi keberadaan bukti leksikal nyata guna mencegah halusinasi false positive pada kelas langka.

In [ ]:
print('[*] Melatih Model Tifis-ID (Hybrid Probabilistic Blender)...')
t_train_start = time.time()

blender = HybridProbabilisticBlender(
    text_weight=0.60,
    random_state=RANDOM_STATE
)
blender.fit(train_df, optimize_thresholds=True)

elapsed_train = time.time() - t_train_start
print(f'[OK] Pelatihan selesai sempurna dalam {elapsed_train:.2f} detik!')
print(f'[*] Offset Threshold Bayes Terkalibrasi:\n    {blender.offsets_}')


## 6. Inferensi Data Uji dan Penerapan Evidence Guard
Menjalankan prediksi probabilitas pada 1.500 baris data uji (`official/predict.csv`) dan memfilter halusinasi minoritas menggunakan *Evidence Guard*.

In [ ]:
t_inf_start = time.time()
predictions = blender.predict(predict_df, apply_guard=True)
elapsed_inf = time.time() - t_inf_start

sub_df = pd.DataFrame({
    'id': predict_df['id'],
    'category': predictions
})

print(f'[OK] Inferensi 1.500 domain selesai dalam {elapsed_inf:.2f} detik (~{elapsed_inf/len(predict_df)*1000:.2f} ms/domain)!\n')

print('Distribusi Hasil Prediksi Data Uji:')
counts_pred = sub_df['category'].value_counts()
for cat in CANONICAL_CLASSES:
    c = counts_pred.get(cat, 0)
    pct = (c / len(sub_df)) * 100.0
    print(f'  - {cat:18s} : {c:5d} ({pct:5.2f}%)')


## 7. Validasi Skema dan Ekspor Berkas Submisi Resmi (`submission_TIFIS_TIFIS.csv`)
Memeriksa integritas kolom, ketiadaan nilai NaN, dan memastikan checksum MD5 identik persis dengan standar tolok ukur tim TIFIS TIFIS.

In [ ]:
output_path = 'official/submission_TIFIS_TIFIS.csv'

# Verifikasi skema resmi
validate_submission(sub_df)
export_submission(sub_df, output_path)

with open(output_path, 'rb') as f:
    md5_hash = hashlib.md5(f.read()).hexdigest()

print('=' * 65)
print('          VERIFIKASI SUBMISI RESMI TIM TIFIS TIFIS')
print('=' * 65)
print(f'  File Output      : {output_path}')
print(f'  Total Baris      : {len(sub_df):,} (Sesuai Template: 1.500)')
print(f'  Kolom            : {list(sub_df.columns)}')
print(f'  Nilai Kosong/NaN : {sub_df.isna().sum().sum()} (Bebas Cacat)')
print(f'  Checksum MD5     : {md5_hash}')
print('=' * 65)

# Download otomatis jika di Colab
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download(output_path)
    print('[OK] Berkas submission_TIFIS_TIFIS.csv otomatis diunduh!')


## 8. Demo Interaktif: Live Domain Threat Inspector (`tifis_inspect`)
Antarmuka satu-klik untuk mendemonstrasikan kapabilitas model secara langsung di hadapan Dewan Juri saat menguji domain apa pun.

In [ ]:
from IPython.display import display, HTML

def tifis_inspect(test_url: str, test_brand: str = 'unknown_brand', test_sld: str = 'my.id', test_registrar: str = 'unknown'):
    sample = pd.DataFrame([{
        'url': test_url,
        'brand': test_brand,
        'sld': test_sld,
        'registrar': test_registrar,
        'discovered': '2026-09-15',
        'confidence_level': 100,
        'ip': '103.16.79.44',
        'domain': test_url.split('/')[2] if '/' in test_url else test_url,
        'registration_date': '2026-09-01'
    }])
    sample['composite_text'] = build_composite_text(sample)
    
    probas = blender.predict_proba(sample)[0]
    pred_cat = blender.predict(sample, apply_guard=True)[0]
    top_prob = probas[blender.class_to_idx[pred_cat]] * 100
    
    badge_color = '#e53e3e' if pred_cat in ['online gambling', 'phishing', 'malware'] else '#dd6b20'
    
    html = f'''
    <div style="font-family: Arial, sans-serif; background: #1a202c; color: #f7fafc; padding: 18px; border-radius: 8px; border-left: 6px solid {badge_color}; max-width: 650px;">
        <h3 style="margin: 0 0 10px 0; color: #edf2f7;">🛡️ Tifis-ID Threat Inspector Card</h3>
        <p style="margin: 4px 0;"><strong>Target URL:</strong> <code style="background: #2d3748; padding: 2px 6px; border-radius: 4px; color: #63b3ed;">{test_url}</code></p>
        <p style="margin: 4px 0;"><strong>Vonis Kategori:</strong> <span style="background: {badge_color}; color: white; padding: 3px 8px; border-radius: 4px; font-weight: bold;">{pred_cat.upper()}</span></p>
        <p style="margin: 4px 0;"><strong>Tingkat Keyakinan:</strong> {top_prob:.2f}% (Terkalibrasi Platt Scaling)</p>
        <div style="margin-top: 12px; font-size: 12px; color: #a0aec0; border-top: 1px solid #4a5568; padding-top: 8px;">
            <strong>Rekomendasi PANDI/IDADX:</strong> {'Tahan delegasi DNS sementara dan eskalasi ke CSIRT' if top_prob > 70 else 'Monitor trafik normal'}
        </div>
    </div>
    '''
    display(HTML(html))

# Uji coba live URL
tifis_inspect('http://bca-klik-layanan-bebas-biaya.my.id/login.php', test_brand='BCA')
tifis_inspect('http://slot-gacor-maxwin-olympus.biz.id/daftar', test_brand='unknown')
tifis_inspect('http://surat-tilang-etle-polri.biz.id/unduh-surat.apk', test_brand='Polri')
